<a href="https://colab.research.google.com/github/alaaguedda/prompt_compress/blob/main/prompt_compress.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install kaggle
!pip install -q kaggle

# Upload kaggle.json
from google.colab import files
files.upload()

# Configure kaggle key
import os, shutil
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
!chmod 600 /root/.kaggle/kaggle.json

# Download dataset
!kaggle datasets download -d frankossai/natural-questions-dataset

# Unzip
!unzip -o natural-questions-dataset.zip

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/frankossai/natural-questions-dataset
License(s): apache-2.0
100% 111M/111M [00:01<00:00, 99.0MB/s]

Archive:  natural-questions-dataset.zip
  inflating: Natural-Questions-Base.csv  
  inflating: Natural-Questions-Filtered.csv  


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/datasets/natural_questions
!cp Natural-Questions-Filtered.csv /content/drive/MyDrive/datasets/natural_questions/

Mounted at /content/drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

path = '/content/drive/MyDrive/datasets/natural_questions/Natural-Questions-Filtered.csv'
df = pd.read_csv(path, low_memory=False)

df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,question,long_answers,short_answers
0,which is the most common use of opt-in e-mail ...,A common example of permission marketing is a ...,A newsletter sent to an advertising firm's cus...
1,how i.met your mother who is the mother,"Tracy McConnell, better known as `` The Mother...",Tracy McConnell
2,who had the most wins in the nfl,Active quarterback Tom Brady holds the records...,Tom Brady
3,who played mantis guardians of the galaxy 2,Pom Klementieff (born May 1986) is a French ac...,Pom Klementieff
4,the nashville sound brought a polished and cos...,"In the early 1960s, the Nashville sound began ...",The use of lush string arrangements with a rea...


In [ ]:
df_small = df.head(10)

In [ ]:
# ============================================================
# CELL 1 — Install (bitsandbytes needed for 4-bit)
# ============================================================
!pip install -q transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.5 MB/s eta 0:00:00


In [ ]:
# ============================================================
# Install required libraries
# ============================================================
!pip install -q rouge-score bert-score sentence-transformers scikit-learn nltk

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.9 MB/s eta 0:00:00


True

In [ ]:
# ============================================================
# CELL 2 — Mount Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_MODEL_PATH = "/content/drive/MyDrive/qwen_7b_4bit"

Mounted at /content/drive


In [ ]:
# ============================================================
# CELL 3 — Load with 4-bit quantization
# ============================================================
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

# This is the critical change — define a quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # NF4 is better than fp4 for LLMs
    bnb_4bit_compute_dtype=torch.float16, # Compute in fp16 even though weights are 4-bit
    bnb_4bit_use_double_quant=True,      # Quantize the quantization constants too — saves ~0.4GB extra
)

if os.path.exists(os.path.join(DRIVE_MODEL_PATH, "config.json")):
    print("✅ Loading from Drive...")
    load_path = DRIVE_MODEL_PATH
else:
    print("⬇️  Downloading from HuggingFace...")
    load_path = MODEL_ID

tokenizer = AutoTokenizer.from_pretrained(load_path, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    load_path,
    quantization_config=bnb_config,   # ← replaces torch_dtype=torch.float16
    device_map="auto",
    trust_remote_code=True
)

# ⚠️ IMPORTANT: 4-bit models cannot be saved with save_pretrained the same way.
# You save the tokenizer and config, but NOT the quantized weights —
# they must be re-quantized on load each time (takes ~3 min, not 10).
# Only save if this is the first run.
if load_path == MODEL_ID:
    print("💾 Saving tokenizer + config to Drive...")
    os.makedirs(DRIVE_MODEL_PATH, exist_ok=True)
    tokenizer.save_pretrained(DRIVE_MODEL_PATH)
    # Save config so the path check works next session
    model.config.save_pretrained(DRIVE_MODEL_PATH)
    print("✅ Saved. Future sessions skip the HuggingFace download (~3 min load instead of ~10 min).")

model.eval()
print("🚀 Model ready.")

# Verify VRAM
if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM after load: {used:.2f} GB / {total:.2f} GB")

⬇️  Downloading from HuggingFace...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

💾 Saving tokenizer + config to Drive...
✅ Saved. Future sessions skip the HuggingFace download (~3 min load instead of ~10 min).
🚀 Model ready.
VRAM after load: 5.58 GB / 15.64 GB


In [ ]:
# ============================================================
# CELL 4 — Generation function (unchanged from 3B version)
# ============================================================
import gc

def generate_answer(question: str, max_new_tokens: int = 100) -> tuple[str, int, int]:
    messages = [
        {
            "role": "system",
            "content": (
                "You are a concise question-answering assistant. "
                "Answer in 1–3 sentences. Do not repeat the question."
            )
        },
        {"role": "user", "content": question}
    ]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )

    new_tokens = outputs[0][input_len:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    output_len = len(new_tokens)

    del inputs, outputs
    torch.cuda.empty_cache()
    gc.collect()

    return answer, input_len, output_len

In [ ]:
# ============================================================
# CELL 5 — Batch runner (same as before)
# ============================================================
import time
import pandas as pd
from tqdm.notebook import tqdm

def run_batch(questions: list[str], label: str = "original") -> pd.DataFrame:
    records = []
    for q in tqdm(questions, desc=f"Generating [{label}]"):
        start = time.time()
        answer, in_tok, out_tok = generate_answer(q)
        elapsed = round(time.time() - start, 2)
        records.append({
            "question":     q,
            "answer":       answer,
            "input_tokens": in_tok,
            "output_tokens":out_tok,
            "time_s":       elapsed,
            "label":        label
        })
    return pd.DataFrame(records)

In [ ]:
# ============================================================
# CELL 6 — Sanity check
# ============================================================
test_questions = [
    "What is the capital of France?",
    "Who wrote Hamlet?",
    "What is the boiling point of water in Celsius?"
]

results = run_batch(test_questions, label="sanity_check")
print(results[["question", "answer", "input_tokens", "output_tokens", "time_s"]])

if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\nVRAM during inference: {used:.2f} GB / {total:.2f} GB")

Generating [sanity_check]:   0%|          | 0/3 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


                                         question  \
0                  What is the capital of France?   
1                               Who wrote Hamlet?   
2  What is the boiling point of water in Celsius?   

                                              answer  input_tokens  \
0                    The capital of France is Paris.            45   
1                  William Shakespeare wrote Hamlet.            43   
2  The boiling point of water in Celsius is 100 d...            48   

   output_tokens  time_s  
0              8    2.63  
1              7    1.18  
2             15    2.19  

VRAM during inference: 5.59 GB / 15.64 GB


In [ ]:
# ============================================================
# Test: Generate answers for first 10 questions
# ============================================================
import time
import pandas as pd
from tqdm.notebook import tqdm

# Take first 10 questions
df_test = df_small.head(10).copy()

qwen_answers = []
times = []

for question in tqdm(df_test["question"], desc="Generating answers"):
    messages = [
        {
            "role": "system",
            "content": "You are a concise question-answering assistant. Answer in 1–3 sentences. Do not repeat the question."
        },
        {"role": "user", "content": question}
    ]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    start = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )
    elapsed = round(time.time() - start, 2)

    new_tokens = outputs[0][input_len:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    qwen_answers.append(answer)
    times.append(elapsed)

    del inputs, outputs
    torch.cuda.empty_cache()
    gc.collect()

# Build df_test_answers
df_test_answers = df_test.copy()
df_test_answers["qwen_answer"] = qwen_answers
df_test_answers["time"] = times

# Preview
print(df_test_answers[["question", "qwen_answer", "time"]])

Generating answers:   0%|          | 0/10 [00:00<?, ?it/s]

                                            question  \
0  which is the most common use of opt-in e-mail ...   
1            how i.met your mother who is the mother   
2                   who had the most wins in the nfl   
3        who played mantis guardians of the galaxy 2   
4  the nashville sound brought a polished and cos...   
5    who needs to be in the car with a permit driver   
6  god's not dead a light in the darkness release...   
7  who is the current president of un general ass...   
8       when do they pull the powerball numbers 2016   
9      what is the name of the sea surrounding dubai   

                                         qwen_answer  time  
0  The most common use of opt-in e-mail marketing...  2.86  
1  In the TV show "How I Met Your Mother," the mo...  3.38  
2  Tom Brady holds the record for the most career...  2.73  
3  Peyton List played Mantis in Guardians of the ...  1.80  
4  The Nashville Sound brought a polished and cos...  4.18  
5  A permit drive

In [ ]:
df_test_answers

,question,long_answers,short_answers,qwen_answer,time
0,which is the most common use of opt-in e-mail ...,A common example of permission marketing is a ...,A newsletter sent to an advertising firm's cus...,The most common use of opt-in e-mail marketing...,2.93
1,how i.met your mother who is the mother,"Tracy McConnell, better known as `` The Mother...",Tracy McConnell,"In the TV show ""How I Met Your Mother,"" the mo...",4.45
2,who had the most wins in the nfl,Active quarterback Tom Brady holds the records...,Tom Brady,Tom Brady holds the record for the most career...,2.19
3,who played mantis guardians of the galaxy 2,Pom Klementieff (born May 1986) is a French ac...,Pom Klementieff,Peyton List played Mantis in Guardians of the ...,1.79
4,the nashville sound brought a polished and cos...,"In the early 1960s, the Nashville sound began ...",The use of lush string arrangements with a rea...,The Nashville Sound brought a polished and cos...,4.27
5,who needs to be in the car with a permit driver,"Typically, a driver operating with a learner's...",An adult licensed driver who is at least 21 ye...,A permit driver must have a licensed adult pas...,4.31
6,god's not dead a light in the darkness release...,Principal photography was completed in Little ...,"March 30, 2018",God's Not Dead: A Light in the Darkness was re...,2.89
7,who is the current president of un general ass...,Miroslaw Lack of Slovakia has been elected as ...,Miroslaw Lack of Slovakia,The current President of the United Nations Ge...,1.88
8,when do they pull the powerball numbers 2016,Drawings for Power ball are held every Wednesd...,Every Wednesday and Saturday evening at 10 : 5...,"In 2016, Powerball drawings were held every We...",1.79
9,what is the name of the sea surrounding dubai,Dubai (/ Dubai / dew - BY ; Arabic : دبي ‎ Dub...,Persian Gulf,The sea surrounding Dubai is the Arabian Sea t...,3.38


In [ ]:
  # ============================================================
  # Similarity metrics
  # ============================================================
  import numpy as np
  import torch
  from rouge_score import rouge_scorer
  from sentence_transformers import SentenceTransformer
  from sklearn.metrics.pairwise import cosine_similarity
  from sklearn.feature_extraction.text import TfidfVectorizer
  from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
  from nltk.tokenize import word_tokenize

  # Load sentence transformer once (for cosine, euclidean, manhattan)
  print("Loading sentence embeddings model...")
  embedder = SentenceTransformer("all-MiniLM-L6-v2")  # lightweight, runs fine on Colab

  references = df_test_answers["long_answers"].tolist()
  predictions = df_test_answers["qwen_answer"].tolist()

  # ── 1. ROUGE (1, 2, L) ───────────────────────────────────────
  scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

  rouge1, rouge2, rougeL = [], [], []
  for ref, pred in zip(references, predictions):
      scores = scorer.score(ref, pred)
      rouge1.append(round(scores["rouge1"].fmeasure, 4))
      rouge2.append(round(scores["rouge2"].fmeasure, 4))
      rougeL.append(round(scores["rougeL"].fmeasure, 4))

  # ── 2. BLEU ───────────────────────────────────────────────────
  smoothie = SmoothingFunction().method4
  bleu_scores = []
  for ref, pred in zip(references, predictions):
      ref_tokens  = word_tokenize(ref.lower())
      pred_tokens = word_tokenize(pred.lower())
      score = sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smoothie)
      bleu_scores.append(round(score, 4))

  # ── 3. TF-IDF Cosine similarity ───────────────────────────────
  tfidf_cosine = []
  vectorizer = TfidfVectorizer()
  for ref, pred in zip(references, predictions):
      try:
          tfidf_matrix = vectorizer.fit_transform([ref, pred])
          score = cosine_similarity(tfidf_matrix[0], tfidf_matrix[1])[0][0]
      except:
          score = 0.0
      tfidf_cosine.append(round(score, 4))

  # ── 4. Sentence embedding metrics ─────────────────────────────
  # Encode all at once for speed
  print("Encoding sentences...")
  ref_embeddings  = embedder.encode(references,  convert_to_numpy=True)
  pred_embeddings = embedder.encode(predictions, convert_to_numpy=True)

  # Cosine similarity (embedding-based)
  emb_cosine = []
  for r, p in zip(ref_embeddings, pred_embeddings):
      score = cosine_similarity([r], [p])[0][0]
      emb_cosine.append(round(float(score), 4))

  # Euclidean distance (lower = more similar)
  euclidean = []
  for r, p in zip(ref_embeddings, pred_embeddings):
      dist = float(np.linalg.norm(r - p))
      euclidean.append(round(dist, 4))

  # Manhattan distance (lower = more similar)
  manhattan = []
  for r, p in zip(ref_embeddings, pred_embeddings):
      dist = float(np.sum(np.abs(r - p)))
      manhattan.append(round(dist, 4))

  # Dot product (higher = more similar)
  dot_product = []
  for r, p in zip(ref_embeddings, pred_embeddings):
      score = float(np.dot(r, p))
      dot_product.append(round(score, 4))

  # ── 5. Jaccard similarity (token overlap) ─────────────────────
  jaccard = []
  for ref, pred in zip(references, predictions):
      ref_set  = set(word_tokenize(ref.lower()))
      pred_set = set(word_tokenize(pred.lower()))
      intersection = ref_set & pred_set
      union        = ref_set | pred_set
      score = len(intersection) / len(union) if union else 0.0
      jaccard.append(round(score, 4))

  # ============================================================
  # Add all metrics as new columns
  # ============================================================
  df_test_answers["rouge1"]       = rouge1
  df_test_answers["rouge2"]       = rouge2
  df_test_answers["rougeL"]       = rougeL
  df_test_answers["bleu"]         = bleu_scores
  df_test_answers["tfidf_cosine"] = tfidf_cosine
  df_test_answers["emb_cosine"]   = emb_cosine
  df_test_answers["euclidean"]    = euclidean
  df_test_answers["manhattan"]    = manhattan
  df_test_answers["dot_product"]  = dot_product
  df_test_answers["jaccard"]      = jaccard

  # ============================================================
  # Preview results
  # ============================================================
  metric_cols = ["question", "rouge1", "rouge2", "rougeL", "bleu",
                "tfidf_cosine", "emb_cosine", "euclidean", "manhattan",
                "dot_product", "jaccard"]

  print(df_test_answers[metric_cols].to_string())

  # Summary statistics
  print("\n── Mean scores across 10 questions ──")
  print(df_test_answers[["rouge1","rouge2","rougeL","bleu",
                          "tfidf_cosine","emb_cosine","euclidean",
                          "manhattan","dot_product","jaccard"]].mean().round(4))

Loading sentence embeddings model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding sentences...
                                                                            question  rouge1  rouge2  rougeL    bleu  tfidf_cosine  emb_cosine  euclidean  manhattan  dot_product  jaccard
0                            which is the most common use of opt-in e-mail marketing  0.3146  0.0690  0.2022  0.0122        0.2961      0.3920     1.1028    17.0897       0.3920   0.1803
1                                            how i.met your mother who is the mother  0.3636  0.1667  0.2182  0.0372        0.5074      0.7110     0.7603    11.7010       0.7110   0.2361
2                                                   who had the most wins in the nfl  0.3896  0.2133  0.3117  0.0186        0.5401      0.8864     0.4766     7.3701       0.8864   0.2979
3                                        who played mantis guardians of the galaxy 2  0.2078  0.1333  0.2078  0.0019        0.3471      0.6233     0.8679    13.4206       0.6233   0.1333
4  the nashville sound brought a polished a

In [ ]:
df_test_answers

,question,long_answers,short_answers,qwen_answer,time,rouge1,rouge2,rougeL,bleu,tfidf_cosine,emb_cosine,euclidean,manhattan,dot_product,jaccard
0,which is the most common use of opt-in e-mail ...,A common example of permission marketing is a ...,A newsletter sent to an advertising firm's cus...,The most common use of opt-in e-mail marketing...,2.93,0.3146,0.0690,0.2022,0.0122,0.2961,0.3920,1.1028,17.0897,0.3920,0.1803
1,how i.met your mother who is the mother,"Tracy McConnell, better known as `` The Mother...",Tracy McConnell,"In the TV show ""How I Met Your Mother,"" the mo...",4.45,0.3636,0.1667,0.2182,0.0372,0.5074,0.7110,0.7603,11.7010,0.7110,0.2361
2,who had the most wins in the nfl,Active quarterback Tom Brady holds the records...,Tom Brady,Tom Brady holds the record for the most career...,2.19,0.3896,0.2133,0.3117,0.0186,0.5401,0.8864,0.4766,7.3701,0.8864,0.2979
3,who played mantis guardians of the galaxy 2,Pom Klementieff (born May 1986) is a French ac...,Pom Klementieff,Peyton List played Mantis in Guardians of the ...,1.79,0.2078,0.1333,0.2078,0.0019,0.3471,0.6233,0.8679,13.4206,0.6233,0.1333
4,the nashville sound brought a polished and cos...,"In the early 1960s, the Nashville sound began ...",The use of lush string arrangements with a rea...,The Nashville Sound brought a polished and cos...,4.27,0.1923,0.0485,0.1346,0.0010,0.4548,0.6752,0.8059,12.5012,0.6752,0.1343
5,who needs to be in the car with a permit driver,"Typically, a driver operating with a learner's...",An adult licensed driver who is at least 21 ye...,A permit driver must have a licensed adult pas...,4.31,0.4507,0.0580,0.2254,0.0270,0.2995,0.7648,0.6858,10.7134,0.7648,0.3400
6,god's not dead a light in the darkness release...,Principal photography was completed in Little ...,"March 30, 2018",God's Not Dead: A Light in the Darkness was re...,2.89,0.2785,0.1558,0.1772,0.0126,0.3894,0.6229,0.8684,13.5763,0.6229,0.2069
7,who is the current president of un general ass...,Miroslaw Lack of Slovakia has been elected as ...,Miroslaw Lack of Slovakia,The current President of the United Nations Ge...,1.88,0.4444,0.2941,0.3889,0.1768,0.3748,0.4210,1.0761,16.9194,0.4210,0.2963
8,when do they pull the powerball numbers 2016,Drawings for Power ball are held every Wednesd...,Every Wednesday and Saturday evening at 10 : 5...,"In 2016, Powerball drawings were held every We...",1.79,0.1391,0.0708,0.1043,0.0000,0.1946,0.6588,0.8261,12.7455,0.6588,0.1176
9,what is the name of the sea surrounding dubai,Dubai (/ Dubai / dew - BY ; Arabic : دبي ‎ Dub...,Persian Gulf,The sea surrounding Dubai is the Arabian Sea t...,3.38,0.2667,0.0752,0.2074,0.0037,0.6317,0.6128,0.8800,13.6352,0.6128,0.1446


In [ ]:
# ============================================================
# CELL 1 — Attention-based token importance scorer
# ============================================================
# Uses DistilBERT (small, fits easily alongside Qwen in VRAM)
# We extract attention weights averaged across all heads and layers,
# then assign each word an importance score.

!pip install -q transformers

import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel

print("Loading DistilBERT for attention scoring...")
attn_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
attn_model     = AutoModel.from_pretrained(
    "distilbert-base-uncased",
    output_attentions=True
)
attn_model.eval()
# Run on CPU to avoid competing with Qwen for VRAM
attn_model = attn_model.to("cpu")
print("Ready.")


def get_word_importance(text: str) -> list[tuple[str, float]]:
    """
    Returns list of (word, importance_score) for each whitespace-separated
    word in `text`. Score = mean attention received by the word's subword
    tokens, averaged across all heads and all layers.
    """
    words = text.split()
    if not words:
        return []

    inputs = attn_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = attn_model(**inputs)

    # outputs.attentions: tuple of (1, num_heads, seq_len, seq_len) per layer
    # Stack → shape: (num_layers, num_heads, seq_len, seq_len)
    attentions = torch.stack(outputs.attentions).squeeze(1)

    # Mean over layers and heads → (seq_len, seq_len)
    # Then mean over the "attending from" axis → (seq_len,) = how much each
    # token is attended TO (i.e. how important it is to others)
    mean_attn = attentions.mean(dim=0).mean(dim=0).mean(dim=0).numpy()
    # mean_attn[i] = average attention token i receives from all other tokens

    # Map subword tokens back to original words
    token_ids   = inputs["input_ids"][0]
    tokens      = attn_tokenizer.convert_ids_to_tokens(token_ids)
    # tokens[0] = [CLS], tokens[-1] = [SEP] — skip them
    tokens      = tokens[1:-1]
    mean_attn   = mean_attn[1:-1]

    # Align subword tokens to words (DistilBERT uses ## prefix for continuations)
    word_scores = []
    word_idx    = 0
    tok_idx     = 0

    while tok_idx < len(tokens) and word_idx < len(words):
        word        = words[word_idx]
        word_lower  = word.lower()
        tok_scores  = []

        # Accumulate subword tokens that belong to this word
        reconstructed = ""
        while tok_idx < len(tokens):
            tok = tokens[tok_idx]
            clean = tok.replace("##", "")
            reconstructed += clean
            tok_scores.append(mean_attn[tok_idx])
            tok_idx += 1
            if reconstructed == word_lower or reconstructed in word_lower[:len(reconstructed)+2]:
                if len(reconstructed) >= len(word_lower) * 0.8:
                    break

        score = float(np.mean(tok_scores)) if tok_scores else 0.0
        word_scores.append((word, score))
        word_idx += 1

    # Fallback: if alignment fails, score remaining words as 0
    while word_idx < len(words):
        word_scores.append((words[word_idx], 0.0))
        word_idx += 1

    return word_scores


def compress_single_pass(text: str, drop_ratio: float) -> str:
    """
    Removes the bottom `drop_ratio` fraction of words by attention score.
    Keeps word order. Minimum 1 word kept.
    """
    word_scores = get_word_importance(text)
    if not word_scores:
        return text

    n_drop = max(0, int(len(word_scores) * drop_ratio))
    n_keep = max(1, len(word_scores) - n_drop)

    # Sort by score, keep top n_keep — then restore original order
    sorted_by_score = sorted(word_scores, key=lambda x: x[1], reverse=True)
    kept_words      = set(id(w) for w in sorted_by_score[:n_keep])

    # Rebuild in original order using index
    result = []
    for i, (word, score) in enumerate(word_scores):
        if i < n_keep or score >= sorted_by_score[n_keep-1][1]:
            result.append(word)
        if len(result) >= n_keep:
            break

    # Cleaner approach: rank by score, mark top-n indices to keep
    scores_only  = [s for _, s in word_scores]
    threshold    = sorted(scores_only, reverse=True)[n_keep - 1]
    kept         = [w for w, s in word_scores if s >= threshold][:n_keep]
    # Preserve original order
    keep_indices = set()
    count        = 0
    for i, (w, s) in enumerate(word_scores):
        if s >= threshold and count < n_keep:
            keep_indices.add(i)
            count += 1

    return " ".join(w for i, (w, _) in enumerate(word_scores) if i in keep_indices)


def compress_two_pass(text: str, drop1: float = 0.10, drop2: float = 0.22) -> str:
    """
    Pass 1: drop `drop1` fraction of words by attention.
    Pass 2: re-score the compressed text, drop `drop2` fraction again.
    """
    after_pass1 = compress_single_pass(text, drop_ratio=drop1)
    after_pass2 = compress_single_pass(after_pass1, drop_ratio=drop2)
    return after_pass2


# ============================================================
# CELL 2 — Apply both strategies to the 10 questions
# ============================================================
from tqdm.notebook import tqdm

questions = df_test_answers["question"].tolist()

compressed_30   = []   # Strategy A: single pass, drop 30%
compressed_2pass = []  # Strategy B: two pass, drop 10% then 15%

for q in tqdm(questions, desc="Compressing questions"):
    c30    = compress_single_pass(q, drop_ratio=0.30)
    c2pass = compress_two_pass(q, drop1=0.10, drop2=0.15)
    compressed_30.append(c30)
    compressed_2pass.append(c2pass)

# Add to dataframe
df_test_answers["compressed_30pct"]   = compressed_30
df_test_answers["compressed_2pass"]   = compressed_2pass

# Also track how many tokens were actually removed
def token_reduction(original, compressed):
    orig_len = len(original.split())
    comp_len = len(compressed.split())
    return round((orig_len - comp_len) / orig_len, 3) if orig_len > 0 else 0.0

df_test_answers["reduction_30pct"]  = [
    token_reduction(o, c)
    for o, c in zip(questions, compressed_30)
]
df_test_answers["reduction_2pass"]  = [
    token_reduction(o, c)
    for o, c in zip(questions, compressed_2pass)
]

# Preview the compression
preview_cols = ["question", "compressed_30pct", "compressed_2pass",
                "reduction_30pct", "reduction_2pass"]
for _, row in df_test_answers[preview_cols].iterrows():
    print(f"\nORIGINAL : {row['question']}")
    print(f"STRAT A  : {row['compressed_30pct']}  (−{row['reduction_30pct']*100:.0f}%)")
    print(f"STRAT B  : {row['compressed_2pass']}  (−{row['reduction_2pass']*100:.0f}%)")
    print("-" * 70)

Loading DistilBERT for attention scoring...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Ready.


Compressing questions:   0%|          | 0/10 [00:00<?, ?it/s]


ORIGINAL : which is the most common use of opt-in e-mail marketing
STRAT A  : which is common use of opt-in marketing  (−30%)
STRAT B  : which is the common use of opt-in marketing  (−20%)
----------------------------------------------------------------------

ORIGINAL : how i.met your mother who is the mother
STRAT A  : how i.met your who is mother  (−25%)
STRAT B  : how i.met your mother who is mother  (−12%)
----------------------------------------------------------------------

ORIGINAL : who had the most wins in the nfl
STRAT A  : who had the most wins nfl  (−25%)
STRAT B  : who had the most wins in nfl  (−12%)
----------------------------------------------------------------------

ORIGINAL : who played mantis guardians of the galaxy 2
STRAT A  : who played mantis guardians galaxy 2  (−25%)
STRAT B  : who played mantis guardians of galaxy 2  (−12%)
----------------------------------------------------------------------

ORIGINAL : the nashville sound brought a polished and cosmopo

In [ ]:
# ============================================================
# CELL 3 — Generate Qwen answers for both compressed versions
# ============================================================
import time
import gc

def answer_compressed(questions_list: list[str], label: str) -> tuple[list[str], list[float]]:
    answers, times = [], []
    for q in tqdm(questions_list, desc=f"Answering [{label}]"):
        messages = [
            {"role": "system", "content": "You are a concise question-answering assistant. Answer in 1–3 sentences."},
            {"role": "user",   "content": q}
        ]
        text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        in_len = inputs["input_ids"].shape[-1]

        start = time.time()
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                use_cache=True,
            )
        elapsed = round(time.time() - start, 2)

        answer = tokenizer.decode(outputs[0][in_len:], skip_special_tokens=True).strip()
        answers.append(answer)
        times.append(elapsed)

        del inputs, outputs
        torch.cuda.empty_cache()
        gc.collect()

    return answers, times


ans_30,    time_30    = answer_compressed(compressed_30,    "30pct single-pass")
ans_2pass, time_2pass = answer_compressed(compressed_2pass, "two-pass")

df_test_answers["qwen_answer_30pct"]    = ans_30
df_test_answers["time_30pct"]           = time_30
df_test_answers["qwen_answer_2pass"]    = ans_2pass
df_test_answers["time_2pass"]           = time_2pass

# ============================================================
# CELL 4 — Final table preview
# ============================================================
print("Columns in df_test_answers:")
print(df_test_answers.columns.tolist())

print("\nSample row (question 0):")
row = df_test_answers.iloc[0]
print(f"  question          : {row['question']}")
print(f"  compressed_30pct  : {row['compressed_30pct']}")
print(f"  compressed_2pass  : {row['compressed_2pass']}")
print(f"  qwen_answer       : {row['qwen_answer']}")
print(f"  qwen_answer_30pct : {row['qwen_answer_30pct']}")
print(f"  qwen_answer_2pass : {row['qwen_answer_2pass']}")

Answering [30pct single-pass]:   0%|          | 0/10 [00:00<?, ?it/s]

Answering [two-pass]:   0%|          | 0/10 [00:00<?, ?it/s]

Columns in df_test_answers:
['question', 'long_answers', 'short_answers', 'qwen_answer', 'time', 'compressed_30pct', 'compressed_2pass', 'reduction_30pct', 'reduction_2pass', 'qwen_answer_30pct', 'time_30pct', 'qwen_answer_2pass', 'time_2pass']

Sample row (question 0):
  question          : which is the most common use of opt-in e-mail marketing
  compressed_30pct  : which is common use of opt-in marketing
  compressed_2pass  : which is the common use of opt-in marketing
  qwen_answer       : The most common use of opt-in e-mail marketing is to send promotional messages and updates to subscribers who have agreed to receive such communications from a business.
  qwen_answer_30pct : Common uses of opt-in marketing include sending targeted email newsletters, promotional offers, and product updates directly to customers who have agreed to receive such communications.
  qwen_answer_2pass : Opt-in marketing is commonly used to gather consent from customers to send them promotional messages,

In [ ]:
# ============================================================
# Full similarity metrics computation & organized DataFrame
# ============================================================
import numpy as np
import torch
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.tokenize import word_tokenize

print("Loading sentence embeddings model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")  # lightweight

# ============================================================
# Define answer variants (keep all in dataset)
# ============================================================
answer_variants = {
    "original": "long_answers",
    "qwen": "qwen_answer",
    "30pct": "qwen_answer_30pct",
    "2pass": "qwen_answer_2pass"
}

# ============================================================
# Encode original answers once
# ============================================================
orig_answers = df_test_answers["long_answers"].tolist()
orig_embeddings = embedder.encode(orig_answers, convert_to_numpy=True)

# Initialize Rouge scorer, BLEU smoothing, TF-IDF
scorer = rouge_scorer.RougeScorer(["rouge1","rouge2","rougeL"], use_stemmer=True)
smoothie = SmoothingFunction().method4
vectorizer = TfidfVectorizer()

# ============================================================
# Compute metrics
# ============================================================
metrics_dict = {}

for var_name, col_name in answer_variants.items():
    preds = df_test_answers[col_name].tolist()

    rouge1, rouge2, rougeL, bleu_scores, tfidf_cosine = [], [], [], [], []
    emb_cosine, euclidean, manhattan, dot_product, jaccard = [], [], [], [], []

    # Encode predictions
    pred_embeddings = embedder.encode(preds, convert_to_numpy=True)

    for i, (ref, pred) in enumerate(zip(orig_answers, preds)):
        # ROUGE
        scores = scorer.score(ref, pred)
        rouge1.append(round(scores["rouge1"].fmeasure, 4))
        rouge2.append(round(scores["rouge2"].fmeasure, 4))
        rougeL.append(round(scores["rougeL"].fmeasure, 4))

        # BLEU
        ref_tokens = word_tokenize(ref.lower())
        pred_tokens = word_tokenize(pred.lower())
        bleu_scores.append(round(sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smoothie),4))

        # TF-IDF cosine
        try:
            tfidf_matrix = vectorizer.fit_transform([ref, pred])
            tfidf_cosine.append(round(cosine_similarity(tfidf_matrix[0], tfidf_matrix[1])[0][0],4))
        except:
            tfidf_cosine.append(0.0)

        # Embedding metrics
        r_emb = orig_embeddings[i]
        p_emb = pred_embeddings[i]

        emb_cosine.append(round(float(cosine_similarity([r_emb],[p_emb])[0][0]),4))
        euclidean.append(round(float(np.linalg.norm(r_emb - p_emb)),4))
        manhattan.append(round(float(np.sum(np.abs(r_emb - p_emb))),4))
        dot_product.append(round(float(np.dot(r_emb, p_emb)),4))

        # Jaccard
        ref_set = set(word_tokenize(ref.lower()))
        pred_set = set(word_tokenize(pred.lower()))
        union = ref_set | pred_set
        jaccard.append(round(len(ref_set & pred_set)/len(union) if union else 0.0,4))

    # Save metrics with variant-specific column names
    metrics_dict.update({
        f"rouge1_{var_name}": rouge1,
        f"rouge2_{var_name}": rouge2,
        f"rougeL_{var_name}": rougeL,
        f"bleu_{var_name}": bleu_scores,
        f"tfidf_cosine_{var_name}": tfidf_cosine,
        f"emb_cosine_{var_name}": emb_cosine,
        f"euclidean_{var_name}": euclidean,
        f"manhattan_{var_name}": manhattan,
        f"dot_product_{var_name}": dot_product,
        f"jaccard_{var_name}": jaccard
    })

# ============================================================
# Add all metrics to DataFrame
# ============================================================
for k, v in metrics_dict.items():
    df_test_answers[k] = v

# ============================================================
# Rearrange columns: questions -> all answers -> metrics by type
# ============================================================
answer_cols = list(answer_variants.values())
metric_types = ["rouge1","rouge2","rougeL","bleu","tfidf_cosine",
                "emb_cosine","euclidean","manhattan","dot_product","jaccard"]

ordered_cols = ["question"] + answer_cols
for metric in metric_types:
    for var_name in answer_variants.keys():
        ordered_cols.append(f"{metric}_{var_name}")

df_test_answers = df_test_answers[ordered_cols]

# ============================================================
# Preview
# ============================================================
print(df_test_answers.head())

Loading sentence embeddings model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


                                            question  \
0  which is the most common use of opt-in e-mail ...   
1            how i.met your mother who is the mother   
2                   who had the most wins in the nfl   
3        who played mantis guardians of the galaxy 2   
4  the nashville sound brought a polished and cos...   

                                        long_answers  \
0  A common example of permission marketing is a ...   
1  Tracy McConnell, better known as `` The Mother...   
2  Active quarterback Tom Brady holds the records...   
3  Pom Klementieff (born May 1986) is a French ac...   
4  In the early 1960s, the Nashville sound began ...   

                                         qwen_answer  \
0  The most common use of opt-in e-mail marketing...   
1  In the TV show "How I Met Your Mother," the mo...   
2  Tom Brady holds the record for the most career...   
3  Peyton List played Mantis in Guardians of the ...   
4  The Nashville Sound brought a polished and 

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
df_test_answers

,question,long_answers,qwen_answer,qwen_answer_30pct,qwen_answer_2pass,rouge1_original,rouge1_qwen,rouge1_30pct,rouge1_2pass,rouge2_original,rouge2_qwen,rouge2_30pct,rouge2_2pass,rougeL_original,rougeL_qwen,rougeL_30pct,rougeL_2pass,bleu_original,bleu_qwen,bleu_30pct,bleu_2pass,tfidf_cosine_original,tfidf_cosine_qwen,tfidf_cosine_30pct,tfidf_cosine_2pass,emb_cosine_original,emb_cosine_qwen,emb_cosine_30pct,emb_cosine_2pass,euclidean_original,euclidean_qwen,euclidean_30pct,euclidean_2pass,manhattan_original,manhattan_qwen,manhattan_30pct,manhattan_2pass,dot_product_original,dot_product_qwen,dot_product_30pct,dot_product_2pass,jaccard_original,jaccard_qwen,jaccard_30pct,jaccard_2pass
0,which is the most common use of opt-in e-mail ...,A common example of permission marketing is a ...,The most common use of opt-in e-mail marketing...,Common uses of opt-in marketing include sendin...,Opt-in marketing is commonly used to gather co...,1.0,0.3146,0.3023,0.3107,1.0,0.0690,0.0238,0.0594,1.0,0.2022,0.2326,0.1942,1.0,0.0122,0.0091,0.0204,1.0,0.2961,0.2634,0.2858,1.0,0.3920,0.4497,0.4640,0.0,1.1028,1.0491,1.0353,0.0,17.0897,16.3283,15.9259,1.0,0.3920,0.4497,0.4640,1.0,0.1803,0.1613,0.1944
1,how i.met your mother who is the mother,"Tracy McConnell, better known as `` The Mother...","In the TV show ""How I Met Your Mother,"" the mo...","""How I Met Your Mother"" is an American televis...","In the TV show ""How I Met Your Mother,"" the ch...",1.0,0.3636,0.4000,0.4677,1.0,0.1667,0.1789,0.1639,1.0,0.2182,0.2400,0.2419,1.0,0.0372,0.0711,0.0611,1.0,0.5074,0.4610,0.5942,1.0,0.7110,0.7090,0.6407,0.0,0.7603,0.7630,0.8477,0.0,11.7010,11.6110,13.2311,1.0,0.7110,0.7090,0.6407,1.0,0.2361,0.2895,0.2857
2,who had the most wins in the nfl,Active quarterback Tom Brady holds the records...,Tom Brady holds the record for the most career...,Tom Brady holds the record for the most career...,Tom Brady holds the record for the most career...,1.0,0.3896,0.5111,0.4359,1.0,0.2133,0.2955,0.1579,1.0,0.3117,0.4222,0.2821,1.0,0.0186,0.0864,0.0157,1.0,0.5401,0.5838,0.5173,1.0,0.8864,0.9084,0.8797,0.0,0.4766,0.4279,0.4905,0.0,7.3701,6.6837,7.7645,1.0,0.8864,0.9084,0.8797,1.0,0.2979,0.3585,0.2979
3,who played mantis guardians of the galaxy 2,Pom Klementieff (born May 1986) is a French ac...,Peyton List played Mantis in Guardians of the ...,Peyton List played the role of Mantis in Guard...,Peyton List played the character Mantis in Gua...,1.0,0.2078,0.2750,0.2051,1.0,0.1333,0.2308,0.1053,1.0,0.2078,0.2750,0.2051,1.0,0.0019,0.0077,0.0010,1.0,0.3471,0.4488,0.3233,1.0,0.6233,0.6176,0.6299,0.0,0.8679,0.8745,0.8604,0.0,13.4206,13.5963,13.3224,1.0,0.6233,0.6176,0.6299,1.0,0.1333,0.1500,0.1311
4,the nashville sound brought a polished and cos...,"In the early 1960s, the Nashville sound began ...",The Nashville Sound brought a polished and cos...,"The Nashville Sound brought a polished, cosmop...","The Nashville Sound brought a polished, cosmop...",1.0,0.1923,0.2400,0.2368,1.0,0.0485,0.0269,0.0265,1.0,0.1346,0.1511,0.1579,1.0,0.0010,0.0044,0.0046,1.0,0.4548,0.4103,0.4353,1.0,0.6752,0.7142,0.6956,0.0,0.8059,0.7561,0.7802,0.0,12.5012,11.8078,12.1230,1.0,0.6752,0.7141,0.6956,1.0,0.1343,0.1631,0.1479
5,who needs to be in the car with a permit driver,"Typically, a driver operating with a learner's...",A permit driver must have a licensed adult pas...,All passengers in the car with a permit driver...,"In most states, only a passenger who has a lea...",1.0,0.4507,0.3457,0.3590,1.0,0.0580,0.0759,0.1053,1.0,0.2254,0.1975,0.2308,1.0,0.0270,0.0298,0.0811,1.0,0.2995,0.1964,0.2422,1.0,0.7648,0.5461,0.8106,0.0,0.6858,0.9528,0.6155,0.0,10.7134,14.6009,9.5958,1.0,0.7648,0.5461,0.8106,1.0,0.3400,0.2131,0.2500
6,god's not dead a light in the darkness release...,Principal photography was completed in Little ...,God's Not Dead: A Light in the Darkness was re...,"""God's Not Dead"" is a series of films that exp...","""God's Not Dead: A Light in the Darkness"" is a...",1.0,0.2785,0.1573,0.2292,1.0,0.155